In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

def process_silver_scd2(tabela_origem: str, tabela_destino: str, chave_primaria: str, colunas_monitoradas: list, batch_id: str):
    print(f"Iniciando processamento da Silver: {tabela_destino}")
    
    # 1. Leitura do Lote Incremental
    df_bronze = spark.read.table(tabela_origem).filter(F.col("batch_id") == batch_id)
    
    # 2. Deduplicação do Lote (Garante que pegamos o último evento caso a mesma conta/cartão venha 2x no batch)
    window_spec = Window.partitionBy(chave_primaria).orderBy(F.col("data_atualizacao").desc())
    df_upsert = df_bronze.withColumn("row_num", F.row_number().over(window_spec)) \
                         .filter(F.col("row_num") == 1).drop("row_num")
                         
    # Adiciona colunas do SCD2
    df_upsert = df_upsert.withColumn("is_active", F.lit(True)) \
                         .withColumn("valid_from", F.col("data_atualizacao")) \
                         .withColumn("valid_to", F.lit(None).cast("string"))

    # 3. MERGE Idempotente
    tabela_existe = spark.catalog.tableExists(tabela_destino)
    
    if not tabela_existe:
        df_upsert.write.format("delta").saveAsTable(tabela_destino)
        print(f"Tabela {tabela_destino} criada com sucesso (Primeira carga).")
    else:
        delta_table = DeltaTable.forName(spark, tabela_destino)
        
        # Constrói dinamicamente a condição de update baseada nas colunas monitoradas
        condicoes_mudanca = " OR ".join([f"target.{col} <> source.{col}" for col in colunas_monitoradas])
        update_cond = f"target.{chave_primaria} = source.{chave_primaria} AND target.is_active = true AND ({condicoes_mudanca})"
        
        # Inativa versão antiga
        delta_table.alias("target").merge(
            df_upsert.alias("source"), update_cond
        ).whenMatchedUpdate(set = {
            "is_active": F.lit(False),
            "valid_to": F.col("source.data_atualizacao")
        }).execute()
        
        # Insere versão nova
        insert_cond = f"target.{chave_primaria} = source.{chave_primaria} AND target.is_active = true"
        delta_table.alias("target").merge(
            df_upsert.alias("source"), insert_cond
        ).whenNotMatchedInsertAll().execute()
        
        print(f"MERGE incremental finalizado para {tabela_destino}.")

# ==========================================
# EXECUÇÃO
# ==========================================
ultimo_batch = spark.read.table("workspace.default.bronze_contas").select("batch_id").limit(1).collect()[0][0]

# Processa Contas (Monitorando mudança de status)
process_silver_scd2(
    tabela_origem="workspace.default.bronze_contas",
    tabela_destino="workspace.default.silver_contas",
    chave_primaria="id_conta",
    colunas_monitoradas=["status_conta", "tipo_conta"],
    batch_id=ultimo_batch
)

# Processa Cartões (Monitorando mudança de status e limite)
process_silver_scd2(
    tabela_origem="workspace.default.bronze_cartoes",
    tabela_destino="workspace.default.silver_cartoes",
    chave_primaria="id_cartao",
    colunas_monitoradas=["status_cartao", "limite"],
    batch_id=ultimo_batch
)